In [ ]:
from pymongo import MongoClient
from pymongo.errors import OperationFailure
from tqdm.auto import tqdm
import math
import random

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

HOST = "10.255.68.40"
PORT = 27017
DB_NAME = "ejoow"

GROUP_COL_NAME = "group_embeddings"
POI_COL_NAME = "poi_embeddings_mlp_v1_a"

SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

client = MongoClient(HOST, PORT)
db = client[DB_NAME]

group_emb_col = db[GROUP_COL_NAME]
poi_emb_col = db[POI_COL_NAME]

print("device:", DEVICE)
print("group_embeddings:", group_emb_col.estimated_document_count())
print("poi_embeddings:", poi_emb_col.estimated_document_count())


In [ ]:
def ensure_index(collection, keys, **kwargs):
    try:
        return collection.create_index(keys, **kwargs)
    except OperationFailure as exc:
        if exc.code == 86:
            print(f"skip conflicting index on {collection.name}: {keys}")
            return None
        raise

ensure_index(group_emb_col, [("VenueID", 1), ("TL", 1)])
ensure_index(group_emb_col, [("cluster_id_k", 1)])
ensure_index(poi_emb_col, "venue_id", unique=True)
ensure_index(poi_emb_col, [("cluster_id", 1)])
ensure_index(poi_emb_col, [("latitude", 1), ("longitude", 1)])
for tl in range(1, 5):
    ensure_index(poi_emb_col, [(f"visit{tl}", 1)])

GROUP_PROJECTION = {
    "_id": 1,
    "Group_ID": 1,
    "VenueID": 1,
    "TL": 1,
    "cluster_id_k": 1,
    "Latitude": 1,
    "Longitude": 1,
    "group_embedding": 1,
    "dim": 1,
}

POI_PROJECTION = {
    "_id": 0,
    "venue_id": 1,
    "category_id": 1,
    "category": 1,
    "cluster_id": 1,
    "latitude": 1,
    "longitude": 1,
    "visit1": 1,
    "visit2": 1,
    "visit3": 1,
    "visit4": 1,
    "poi_embedding": 1,
    "embedding_dim": 1,
}


In [ ]:
def to_np_vector(values):
    arr = np.asarray(values, dtype=np.float32)
    if arr.ndim != 1:
        raise ValueError(f"expected 1D vector, got shape={arr.shape}")
    return arr

def to_optional_int(value):
    try:
        return None if value is None else int(value)
    except (TypeError, ValueError):
        return None

def tl_visit_field(tl):
    tl = int(tl)
    if tl not in (1, 2, 3, 4):
        raise ValueError(f"unsupported TL: {tl}")
    return f"visit{tl}"

def bbox_from_radius_km(latitude, radius_km):
    lat_margin = radius_km / 111.32
    cos_lat = max(math.cos(math.radians(float(latitude))), 1e-6)
    lon_margin = radius_km / (111.32 * cos_lat)
    return lat_margin, lon_margin

def haversine_km(lat1, lon1, lat2, lon2):
    lat1 = math.radians(float(lat1))
    lon1 = math.radians(float(lon1))
    lat2 = math.radians(float(lat2))
    lon2 = math.radians(float(lon2))

    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = math.sin(dlat / 2.0) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2.0) ** 2
    return 6371.0088 * (2.0 * math.asin(math.sqrt(a)))

positive_poi_cache = {}
negative_candidate_cache = {}


In [ ]:
def find_group_sample(extra_query=None):
    query = {
        "group_embedding": {"$exists": True},
        "VenueID": {"$exists": True, "$ne": None},
        "TL": {"$in": [1, 2, 3, 4]},
        "Latitude": {"$exists": True, "$ne": None},
        "Longitude": {"$exists": True, "$ne": None},
    }
    if extra_query:
        query.update(extra_query)

    doc = group_emb_col.find_one(query, GROUP_PROJECTION)
    if doc is None:
        raise RuntimeError("group_embeddings 에서 학습 가능한 샘플을 찾지 못했습니다.")
    return doc

def load_positive_poi(group_doc):
    venue_id = str(group_doc["VenueID"])
    cached = positive_poi_cache.get(venue_id)
    if cached is not None:
        return cached

    doc = poi_emb_col.find_one(
        {"venue_id": venue_id, "poi_embedding": {"$exists": True}},
        POI_PROJECTION,
    )
    if doc is None:
        raise ValueError(f"positive POI embedding not found: venue_id={venue_id}")

    positive_poi_cache[venue_id] = doc
    return doc

def fetch_negative_candidates(group_doc, positive_venue_id, radius_km=5.0, limit=300):
    cache_key = (str(group_doc["_id"]), float(radius_km), int(limit))
    cached = negative_candidate_cache.get(cache_key)
    if cached is not None:
        return cached

    tl = int(group_doc["TL"])
    cluster_id = to_optional_int(group_doc.get("cluster_id_k"))
    latitude = float(group_doc["Latitude"])
    longitude = float(group_doc["Longitude"])
    visit_field = tl_visit_field(tl)
    lat_margin, lon_margin = bbox_from_radius_km(latitude, radius_km)

    or_filters = [{visit_field: {"$gt": 0}}]
    if cluster_id is not None:
        or_filters.append({"cluster_id": cluster_id})
    or_filters.append(
        {
            "latitude": {"$gte": latitude - lat_margin, "$lte": latitude + lat_margin},
            "longitude": {"$gte": longitude - lon_margin, "$lte": longitude + lon_margin},
        }
    )

    query = {
        "venue_id": {"$ne": str(positive_venue_id)},
        "poi_embedding": {"$exists": True},
        "$or": or_filters,
    }

    seen = set()
    candidates = []
    cursor = poi_emb_col.find(query, POI_PROJECTION).limit(limit * 5)

    for doc in cursor:
        venue_id = str(doc["venue_id"])
        if venue_id in seen:
            continue
        seen.add(venue_id)

        same_tl = int(doc.get(visit_field, 0) or 0) > 0
        same_cluster = cluster_id is not None and to_optional_int(doc.get("cluster_id")) == cluster_id

        if doc.get("latitude") is None or doc.get("longitude") is None:
            distance_km = float("inf")
            nearby = False
        else:
            distance_km = haversine_km(latitude, longitude, doc["latitude"], doc["longitude"])
            nearby = distance_km <= radius_km

        if not (same_tl or same_cluster or nearby):
            continue

        item = dict(doc)
        item["same_tl"] = same_tl
        item["same_cluster"] = same_cluster
        item["nearby"] = nearby
        item["distance_km"] = float(distance_km)
        item["priority"] = int(same_tl) + int(same_cluster) + int(nearby)
        candidates.append(item)

    candidates.sort(key=lambda x: (-x["priority"], x["distance_km"], x["venue_id"]))

    if len(candidates) < limit:
        fallback_query = {
            "venue_id": {"$ne": str(positive_venue_id)},
            "poi_embedding": {"$exists": True},
        }
        fallback_cursor = poi_emb_col.find(fallback_query, POI_PROJECTION).limit(limit)

        for doc in fallback_cursor:
            venue_id = str(doc["venue_id"])
            if venue_id in seen:
                continue
            seen.add(venue_id)

            if doc.get("latitude") is None or doc.get("longitude") is None:
                distance_km = float("inf")
            else:
                distance_km = haversine_km(latitude, longitude, doc["latitude"], doc["longitude"])

            item = dict(doc)
            item["same_tl"] = False
            item["same_cluster"] = False
            item["nearby"] = False
            item["distance_km"] = float(distance_km)
            item["priority"] = 0
            candidates.append(item)

            if len(candidates) >= limit:
                break

    if not candidates:
        raise ValueError(f"negative candidate not found for venue_id={positive_venue_id}")

    negative_candidate_cache[cache_key] = candidates[:limit]
    return negative_candidate_cache[cache_key]

def sample_negative_poi(group_doc, positive_venue_id, radius_km=5.0, limit=300, topk_random=30):
    candidates = fetch_negative_candidates(
        group_doc,
        positive_venue_id=positive_venue_id,
        radius_km=radius_km,
        limit=limit,
    )
    topk = min(len(candidates), int(topk_random))
    sampled = random.choice(candidates[:topk])
    return sampled, candidates

def build_triplet_from_group(group_doc, radius_km=5.0, limit=300, topk_random=30):
    positive_doc = load_positive_poi(group_doc)
    negative_doc, candidates = sample_negative_poi(
        group_doc,
        positive_venue_id=positive_doc["venue_id"],
        radius_km=radius_km,
        limit=limit,
        topk_random=topk_random,
    )
    return {
        "group_doc": group_doc,
        "positive_doc": positive_doc,
        "negative_doc": negative_doc,
        "negative_candidates": candidates,
        "group_vec": to_np_vector(group_doc["group_embedding"]),
        "positive_vec": to_np_vector(positive_doc["poi_embedding"]),
        "negative_vec": to_np_vector(negative_doc["poi_embedding"]),
    }


In [ ]:
sample_group = find_group_sample()
sample_triplet = build_triplet_from_group(
    sample_group,
    radius_km=5.0,
    limit=200,
    topk_random=20,
)

print("sample group _id:", sample_group["_id"])
print("Group_ID:", sample_group.get("Group_ID"))
print("positive venue_id:", sample_triplet["positive_doc"]["venue_id"])
print("negative venue_id:", sample_triplet["negative_doc"]["venue_id"])
print("TL:", sample_group["TL"])
print("group cluster_id_k:", sample_group.get("cluster_id_k"))
print("negative flags:", {
    "same_tl": sample_triplet["negative_doc"]["same_tl"],
    "same_cluster": sample_triplet["negative_doc"]["same_cluster"],
    "nearby": sample_triplet["negative_doc"]["nearby"],
    "distance_km": round(sample_triplet["negative_doc"]["distance_km"], 3),
})
print("group dim:", len(sample_triplet["group_vec"]))
print("poi dim:", len(sample_triplet["positive_vec"]))
print("num negative candidates:", len(sample_triplet["negative_candidates"]))

[
    {
        "venue_id": doc["venue_id"],
        "priority": doc["priority"],
        "same_tl": doc["same_tl"],
        "same_cluster": doc["same_cluster"],
        "nearby": doc["nearby"],
        "distance_km": round(doc["distance_km"], 3),
    }
    for doc in sample_triplet["negative_candidates"][:5]
]


In [ ]:
class GroupPoiScorer(nn.Module):
    def __init__(self, group_dim, poi_dim, hidden_dim=128, dropout=0.1):
        super().__init__()
        self.group_proj = nn.Sequential(
            nn.Linear(group_dim, hidden_dim),
            nn.ReLU(),
            nn.LayerNorm(hidden_dim),
            nn.Dropout(dropout),
        )
        self.poi_proj = nn.Sequential(
            nn.Linear(poi_dim, hidden_dim),
            nn.ReLU(),
            nn.LayerNorm(hidden_dim),
            nn.Dropout(dropout),
        )
        self.head = nn.Sequential(
            nn.Linear(hidden_dim * 4, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, group_emb, poi_emb):
        g = F.normalize(self.group_proj(group_emb), p=2, dim=-1)
        p = F.normalize(self.poi_proj(poi_emb), p=2, dim=-1)
        pair = torch.cat([g, p, g * p, torch.abs(g - p)], dim=-1)
        return self.head(pair).squeeze(-1)

def bpr_loss(pos_score, neg_score):
    return -F.logsigmoid(pos_score - neg_score).mean()

group_dim = sample_triplet["group_vec"].shape[0]
poi_dim = sample_triplet["positive_vec"].shape[0]

model = GroupPoiScorer(group_dim=group_dim, poi_dim=poi_dim, hidden_dim=128, dropout=0.1).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)

print(model)


In [ ]:
group_tensor = torch.tensor(sample_triplet["group_vec"], dtype=torch.float32, device=DEVICE).unsqueeze(0)
pos_tensor = torch.tensor(sample_triplet["positive_vec"], dtype=torch.float32, device=DEVICE).unsqueeze(0)
neg_tensor = torch.tensor(sample_triplet["negative_vec"], dtype=torch.float32, device=DEVICE).unsqueeze(0)

model.train()
optimizer.zero_grad()

pos_score = model(group_tensor, pos_tensor)
neg_score = model(group_tensor, neg_tensor)
loss = bpr_loss(pos_score, neg_score)

loss.backward()
optimizer.step()

print("pos_score:", float(pos_score.item()))
print("neg_score:", float(neg_score.item()))
print("bpr_loss:", float(loss.item()))


In [ ]:
def load_train_groups(limit=2048):
    query = {
        "group_embedding": {"$exists": True},
        "VenueID": {"$exists": True, "$ne": None},
        "TL": {"$in": [1, 2, 3, 4]},
        "Latitude": {"$exists": True, "$ne": None},
        "Longitude": {"$exists": True, "$ne": None},
    }

    group_docs = list(group_emb_col.find(query, GROUP_PROJECTION).limit(limit))
    if not group_docs:
        return []

    venue_ids = sorted({str(doc["VenueID"]) for doc in group_docs})
    positive_docs = poi_emb_col.find(
        {"venue_id": {"$in": venue_ids}, "poi_embedding": {"$exists": True}},
        {"_id": 0, "venue_id": 1},
    )
    valid_venue_ids = {str(doc["venue_id"]) for doc in positive_docs}

    return [doc for doc in group_docs if str(doc["VenueID"]) in valid_venue_ids]

def make_batch_triplets(group_docs, radius_km=5.0, candidate_limit=300, topk_random=30):
    batch_items = []
    for doc in group_docs:
        try:
            item = build_triplet_from_group(
                doc,
                radius_km=radius_km,
                limit=candidate_limit,
                topk_random=topk_random,
            )
            batch_items.append(item)
        except Exception:
            continue
    return batch_items

TRAIN_CONFIG = {
    "max_groups": 1024,
    "epochs": 3,
    "batch_size": 32,
    "radius_km": 5.0,
    "candidate_limit": 200,
    "topk_random": 20,
}

train_groups = load_train_groups(limit=TRAIN_CONFIG["max_groups"])
print("loaded train groups:", len(train_groups))

history = []
for epoch in range(TRAIN_CONFIG["epochs"]):
    random.shuffle(train_groups)
    epoch_losses = []
    skipped = 0

    progress = tqdm(
        range(0, len(train_groups), TRAIN_CONFIG["batch_size"]),
        desc=f"Epoch {epoch + 1}/{TRAIN_CONFIG['epochs']}",
    )

    for start in progress:
        batch_docs = train_groups[start : start + TRAIN_CONFIG["batch_size"]]
        batch_items = make_batch_triplets(
            batch_docs,
            radius_km=TRAIN_CONFIG["radius_km"],
            candidate_limit=TRAIN_CONFIG["candidate_limit"],
            topk_random=TRAIN_CONFIG["topk_random"],
        )
        skipped += len(batch_docs) - len(batch_items)

        if not batch_items:
            continue

        group_batch = torch.tensor(
            np.stack([item["group_vec"] for item in batch_items]),
            dtype=torch.float32,
            device=DEVICE,
        )
        pos_batch = torch.tensor(
            np.stack([item["positive_vec"] for item in batch_items]),
            dtype=torch.float32,
            device=DEVICE,
        )
        neg_batch = torch.tensor(
            np.stack([item["negative_vec"] for item in batch_items]),
            dtype=torch.float32,
            device=DEVICE,
        )

        model.train()
        optimizer.zero_grad()

        pos_scores = model(group_batch, pos_batch)
        neg_scores = model(group_batch, neg_batch)
        loss = bpr_loss(pos_scores, neg_scores)

        loss.backward()
        optimizer.step()

        loss_value = float(loss.item())
        epoch_losses.append(loss_value)
        progress.set_postfix(loss=f"{loss_value:.4f}", valid=len(batch_items), skipped=skipped)

    mean_loss = float(np.mean(epoch_losses)) if epoch_losses else float("nan")
    history.append({"epoch": epoch + 1, "mean_loss": mean_loss, "skipped": skipped})
    print(f"epoch={epoch + 1} mean_loss={mean_loss:.4f} skipped={skipped}")

history


In [ ]:
@torch.inference_mode()
def recommend_for_group(model, group_doc, radius_km=5.0, limit=200, topk=10):
    positive_venue_id = str(group_doc["VenueID"])
    candidates = fetch_negative_candidates(
        group_doc,
        positive_venue_id=positive_venue_id,
        radius_km=radius_km,
        limit=limit,
    )
    if not candidates:
        return []

    group_vec = torch.tensor(
        to_np_vector(group_doc["group_embedding"]),
        dtype=torch.float32,
        device=DEVICE,
    ).unsqueeze(0)
    poi_batch = torch.tensor(
        np.stack([to_np_vector(doc["poi_embedding"]) for doc in candidates]),
        dtype=torch.float32,
        device=DEVICE,
    )
    group_batch = group_vec.expand(poi_batch.shape[0], -1)

    model.eval()
    scores = model(group_batch, poi_batch).detach().cpu().numpy()

    ranked = []
    for doc, score in sorted(zip(candidates, scores), key=lambda x: float(x[1]), reverse=True)[:topk]:
        ranked.append(
            {
                "venue_id": doc["venue_id"],
                "category": doc.get("category"),
                "score": float(score),
                "same_tl": bool(doc["same_tl"]),
                "same_cluster": bool(doc["same_cluster"]),
                "nearby": bool(doc["nearby"]),
                "distance_km": float(doc["distance_km"]),
            }
        )
    return ranked

recommendations = recommend_for_group(
    model,
    sample_group,
    radius_km=TRAIN_CONFIG["radius_km"],
    limit=TRAIN_CONFIG["candidate_limit"],
    topk=10,
)
recommendations
